In [ ]:
3
!pip install nba_api
from nba_api.stats.endpoints import playergamelog, teamgamelogs, commonplayerinfo
from nba_api.stats.static import players, teams
import pandas as pd
import logging
import sys

# Setup logging for better visibility
logging.basicConfig(level=logging.INFO, format='%(message)s')

# Function to get Player ID by Partial Name (Active Players Only)
def get_player_id_by_partial_name(player_name):
    all_players = players.get_active_players()
    matching_players = [p for p in all_players if player_name.lower() in p['full_name'].lower()]
    if not matching_players:
        raise ValueError(f"No active player found with the name '{player_name}'.")
    if len(matching_players) > 12:
        print("Too many results found. Showing the first 12 matches:")
        matching_players = matching_players[:12]
    if len(matching_players) > 1:
        print("\nMultiple active players found:")
        for idx, p in enumerate(matching_players, 1):
            print(f"{idx}. {p['full_name']}")
        choice = input("Enter the number of the player you want: ").strip()
        if choice.isdigit() and 1 <= int(choice) <= len(matching_players):
            selected = matching_players[int(choice) - 1]
            return selected['id'], selected['full_name']
        else:
            raise ValueError("Invalid selection. Please try again.")
    else:
        return matching_players[0]['id'], matching_players[0]['full_name']

# Function to fetch Player Game Logs
def fetch_player_data_by_name(player_name, season="2024-25"):
    player_id, full_name = get_player_id_by_partial_name(player_name)
    logging.info(f"\nFetching game logs for: {full_name}")
    gamelog = playergamelog.PlayerGameLog(
        player_id=player_id,
        season=season,
        season_type_all_star="Regular Season"  # Correct parameter for PlayerGameLog
    )
    df = gamelog.get_data_frames()[0]
    return df, full_name, player_id

# Function to map Team Abbreviation to Team ID
def get_team_id_by_abbreviation(abbreviation):
    nba_teams = teams.get_teams()
    team = next((team for team in nba_teams if team['abbreviation'] == abbreviation), None)
    return team['id'] if team else None

# Function to find Team Name by Team ID
def find_team_name_by_id(team_id):
    nba_teams = teams.get_teams()
    team = next((team for team in nba_teams if team['id'] == team_id), None)
    return team['full_name'] if team else None

# Function to get Opponent Team ID from the Game Logs
def get_opponent_team_id(df):
    # Extract opponent team abbreviations from MATCHUP column
    opponent_abbreviations = df['MATCHUP'].apply(lambda x: x.split(' ')[-1]).unique()
    opponent_team_ids = []
    for abbrev in opponent_abbreviations:
        team_id = get_team_id_by_abbreviation(abbrev)
        if team_id:
            opponent_team_ids.append(team_id)
    if not opponent_team_ids:
        print("No opponent teams found in the game logs.")
        return None
    print("\nRecent Opponent Teams:")
    unique_opponents = list(set(opponent_team_ids))
    for idx, team_id in enumerate(unique_opponents, 1):
        team_name = find_team_name_by_id(team_id)
        print(f"{idx}. {team_name} ({team_id})")
    choice = input("Choose an opponent team number for strength analysis (or press Enter to skip): ").strip()
    if choice.isdigit() and 1 <= int(choice) <= len(unique_opponents):
        return unique_opponents[int(choice) - 1]  # Return Team ID
    return None

# Function to get Opponent Team Name from Team ID
def get_team_name_by_id(team_id):
    team_name = find_team_name_by_id(team_id)
    return team_name if team_name else "Unknown Team"

# Function to fetch recent opponent team stats
def get_opponent_team_stats(team_id, rankings, season="2024-25", num_games=5):
    # Fetch recent team game logs
    logs = teamgamelogs.TeamGameLogs(
        team_id_nullable=team_id,
        season_nullable=season,
        season_type_nullable="Regular Season"  # Correct parameter for TeamGameLogs
    )
    df = logs.get_data_frames()[0]

    if df.empty:
        return None

    # Filter only numeric columns for calculations
    numeric_stats = ['PTS', 'REB', 'AST', 'STL', 'BLK']
    team_games = df[numeric_stats].head(num_games).apply(pd.to_numeric, errors='coerce')

    # Calculate averages for last 'num_games'
    team_avg = team_games.mean()

    # Add rankings
    team_rankings = {stat: rankings[stat].get(team_id, 'N/A') for stat in numeric_stats}

    team_avg = team_avg.round(1).to_dict()
    team_avg.update({f"{stat}_RANK": rank for stat, rank in team_rankings.items()})

    return team_avg

# Function to calculate league-wide team rankings
def get_league_team_rankings(season="2024-25"):
    logging.info("\nFetching league-wide team statistics for rankings...")
    logs = teamgamelogs.TeamGameLogs(
        season_nullable=season,
        season_type_nullable="Regular Season"  # Correct parameter for TeamGameLogs
    )
    df = logs.get_data_frames()[0]

    # Aggregate stats per team
    aggregated = df.groupby('TEAM_ID').agg({
        'PTS': 'mean',
        'REB': 'mean',
        'AST': 'mean',
        'STL': 'mean',
        'BLK': 'mean'
    }).reset_index()

    # Calculate rankings
    stats = ['PTS', 'REB', 'AST', 'STL', 'BLK']
    rankings = {}
    for stat in stats:
        aggregated[f'{stat}_RANK'] = aggregated[stat].rank(ascending=False, method='min')
        rankings[stat] = dict(zip(aggregated['TEAM_ID'], aggregated[f'{stat}_RANK']))

    logging.info("League-wide team rankings calculated.")
    return rankings

# Function to calculate weighted averages of player stats
def calculate_averages(df, num_games=5):
    stats = ['PTS', 'REB', 'AST', 'STL', 'BLK']
    stats = [stat for stat in stats if stat in df.columns]
    df[stats + ['MIN']] = df[stats + ['MIN']].apply(pd.to_numeric, errors='coerce')
    avg_minutes = df['MIN'].mean()

    def weight_game(row):
        if row['MIN'] < 0.7 * avg_minutes:
            return 0.5
        elif row['MIN'] > 0.85 * avg_minutes:
            return 1.5
        return 1.0

    df['WEIGHT'] = df.apply(weight_game, axis=1)
    recent_games = df.head(num_games)
    weighted_avg = (recent_games[stats].multiply(recent_games['WEIGHT'], axis=0)).sum() / recent_games['WEIGHT'].sum()
    p = weighted_avg.get('PTS', 0)
    r = weighted_avg.get('REB', 0)
    a = weighted_avg.get('AST', 0)
    p_r = p + r
    p_a = p + a
    r_a = r + a
    p_r_a = p + r + a
    return {
        'PTS': round(p, 1), 'REB': round(r, 1), 'AST': round(a, 1),
        'STL': round(weighted_avg.get('STL', 0), 1), 'BLK': round(weighted_avg.get('BLK', 0), 1),
        'P+R': round(p_r, 1), 'P+A': round(p_a, 1), 'R+A': round(r_a, 1), 'P+R+A': round(p_r_a, 1)
    }

from nba_api.stats.endpoints import commonteamroster

# Function to get player position with standardized codes
def get_player_position(player_id, season="2024-25"):
    try:
        info = commonplayerinfo.CommonPlayerInfo(player_id=player_id)
        data = info.get_data_frames()[0]
        position_full = data['POSITION'][0].strip()

        # Mapping full position names to shorthand codes
        position_map = {
            'G': 'G',
            'SG': 'SG',
            'Shooting Guard': 'SG',
            'SF': 'SF',
            'Shooting Forward': 'SF',
            'PF': 'PF',
            'Power Forward': 'PF',
            'C': 'C',
            'Center': 'C',
            '': '',
            None: ''
        }

        # Handle cases where 'POSITION' might be in full name or shorthand
        position_shorthand = position_map.get(position_full, '')
        if not position_shorthand:
            # Attempt to derive shorthand from first letter if possible
            if position_full.upper() in ['G', 'SG', 'SF', 'PF', 'C']:
                position_shorthand = position_full.upper()
            else:
                position_shorthand = ''

        return position_shorthand
    except Exception as e:
        logging.error(f"Error fetching position for player ID {player_id}: {e}")
        return None

# Function to calculate matchup deltas
def get_matchup_deltas(opponent_team_id, position, season="2024-25"):
    logging.info(f"\nCalculating matchup deltas for position: {position} against {get_team_name_by_id(opponent_team_id)}...")
    all_players = players.get_active_players()
    # Safely access 'position' key and compare
    same_position_players = [p for p in all_players if p.get('position', '') == position]

    deltas = {'PTS': [], 'REB': [], 'AST': [], 'STL': [], 'BLK': []}

    opponent_team_name = get_team_name_by_id(opponent_team_id)
    if opponent_team_name is None:
        logging.error("Opponent team name could not be determined.")
        return deltas

    for idx, player in enumerate(same_position_players, 1):
        player_id = player['id']
        try:
            gamelog = playergamelog.PlayerGameLog(
                player_id=player_id,
                season=season,
                season_type_all_star="Regular Season"  # Correct parameter for PlayerGameLog
            )
            df = gamelog.get_data_frames()[0]
            if df.empty:
                continue
            # Games against the opponent
            matchup_games = df[df['MATCHUP'].str.contains(f"@ {opponent_team_name}|vs. {opponent_team_name}")]
            if matchup_games.empty:
                continue
            # Calculate average stats in matchup games
            matchup_avg = matchup_games[['PTS', 'REB', 'AST', 'STL', 'BLK']].mean()
            # Calculate season averages
            season_avg = df[['PTS', 'REB', 'AST', 'STL', 'BLK']].mean()
            # Calculate deltas
            delta = matchup_avg - season_avg
            for stat in deltas.keys():
                deltas[stat].append(delta.get(stat, 0))
        except Exception as e:
            logging.error(f"Error processing player {player['full_name']}: {e}")
            continue

    # Calculate average deltas
    avg_deltas = {stat: round(pd.Series(values).mean(), 2) if values else 0 for stat, values in deltas.items()}
    logging.info("Matchup deltas calculated.")
    return avg_deltas

# Function to adjust projected stats based on matchup deltas
def calculate_projected_stats(projected_line, matchup_deltas):
    projected = {}
    for stat, value in projected_line.items():
        # Only adjust base stats, not combined stats like P+R
        if stat in matchup_deltas:
            projected[stat] = round(value + matchup_deltas[stat], 1)
        else:
            projected[stat] = value
    return projected

# Main Program
if __name__ == "__main__":
    # Fetch league-wide team rankings once to avoid redundant API calls
    try:
        rankings = get_league_team_rankings(season="2024-25")
    except Exception as e:
        logging.error(f"Failed to fetch league team rankings: {e}")
        sys.exit(1)

    while True:
        player_name = input("Enter the player's name or last name (or type 'quit' to exit): ").strip()
        if player_name.lower() in ["quit", "exit"]:
            print("Exiting the program.")
            break

        try:
            num_games_input = input("Enter the number of recent games to analyze (default is 5): ").strip()
            num_games = int(num_games_input) if num_games_input.isdigit() else 5

            season = "2024-25"
            df, full_name, player_id = fetch_player_data_by_name(player_name, season)

            if df.empty:
                print(f"No game logs found for {full_name} in the {season} season.")
                continue

            opponent_team = get_opponent_team_id(df)
            matchup_deltas = {}
            position = None

            if opponent_team:
                opponent_stats = get_opponent_team_stats(opponent_team, rankings, season=season, num_games=num_games)
                if opponent_stats is not None:
                    print("\nOpponent Team Recent Averages and Rankings:")
                    for stat, value in opponent_stats.items():
                        print(f"{stat}: {value}")
                else:
                    print("Could not fetch opponent team stats.")

                # Get player's position
                position = get_player_position(player_id, season=season)
                if position:
                    matchup_deltas = get_matchup_deltas(opponent_team, position, season=season)
                    print("\nMatchup Deltas:")
                    for stat, delta in matchup_deltas.items():
                        print(f"{stat}: {delta}")
                else:
                    print("Could not determine player position.")
            else:
                print("No opponent team selected for strength analysis.")

            # Calculate projected averages
            projected_line = calculate_averages(df, num_games=num_games)

            # Adjust projections based on matchup deltas
            if opponent_team and position and matchup_deltas:
                projected_line = calculate_projected_stats(projected_line, matchup_deltas)

            print(f"\nProjected line for {full_name} based on last {num_games} games and matchup:")
            for stat, value in projected_line.items():
                print(f"{stat}: {value}")

            # Save projections to CSV
            file_name = f"{full_name.replace(' ', '_')}_projected_stats.csv"
            pd.DataFrame([projected_line]).to_csv(file_name, index=False)
            print(f"Projected stats saved to '{file_name}'.")

        except ValueError as e:
            print(e)
        except Exception as e:
            logging.error(f"An error occurred: {e}")
        print("\n--------------------------------------\n")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.9/284.9 kB 5.6 MB/s eta 0:00:00
Enter the player's name or last name (or type 'quit' to exit): 5
Enter the number of recent games to analyze (default is 5): ja morant
No active player found with the name '5'.

--------------------------------------

Enter the player's name or last name (or type 'quit' to exit): morant
Enter the number of recent games to analyze (default is 5): 5

Recent Opponent Teams:
1. Boston Celtics (1610612738)
2. Cleveland Cavaliers (1610612739)
3. New Orleans Pelicans (1610612740)
4. Dallas Mavericks (1610612742)
5. Golden State Warriors (1610612744)
6. Houston Rockets (1610612745)
7. Los Angeles Clippers (1610612746)
8. Los Angeles Lakers (1610612747)
9. Milwaukee Bucks (1610612749)
10. Minnesota Timberwolves (1610612750)
11. Brooklyn Nets (1610612751)
12. New York Knicks (1610612752)
13. Orlando Magic (1610612753)
14. Indiana Pacers (1610612754)
15. Philadelphia 76ers (1610612755)
16. Phoenix Suns (1610612756)
17

In [ ]:

!pip install nba_api
from nba_api.stats.endpoints import playergamelog, teamgamelogs, commonplayerinfo
from nba_api.stats.static import players, teams
import pandas as pd
import logging
import sys

# Setup logging for better visibility
logging.basicConfig(level=logging.INFO, format='%(message)s')

# Function to get Player ID by Partial Name (Active Players Only)
def get_player_id_by_partial_name(player_name):
    all_players = players.get_active_players()
    matching_players = [p for p in all_players if player_name.lower() in p['full_name'].lower()]
    if not matching_players:
        raise ValueError(f"No active player found with the name '{player_name}'.")
    if len(matching_players) > 12:
        print("Too many results found. Showing the first 12 matches:")
        matching_players = matching_players[:12]
    if len(matching_players) > 1:
        print("\nMultiple active players found:")
        for idx, p in enumerate(matching_players, 1):
            print(f"{idx}. {p['full_name']}")
        choice = input("Enter the number of the player you want: ").strip()
        if choice.isdigit() and 1 <= int(choice) <= len(matching_players):
            selected = matching_players[int(choice) - 1]
            return selected['id'], selected['full_name']
        else:
            raise ValueError("Invalid selection. Please try again.")
    else:
        return matching_players[0]['id'], matching_players[0]['full_name']

# Function to fetch Player Game Logs
def fetch_player_data_by_name(player_name, season="2024-25"):
    player_id, full_name = get_player_id_by_partial_name(player_name)
    logging.info(f"\nFetching game logs for: {full_name}")
    gamelog = playergamelog.PlayerGameLog(
        player_id=player_id,
        season=season,
        season_type_all_star="Regular Season"  # Correct parameter for PlayerGameLog
    )
    df = gamelog.get_data_frames()[0]
    return df, full_name, player_id

# Function to map Team Abbreviation to Team ID
def get_team_id_by_abbreviation(abbreviation):
    nba_teams = teams.get_teams()
    team = next((team for team in nba_teams if team['abbreviation'] == abbreviation), None)
    return team['id'] if team else None

# Function to find Team Name by Team ID
def find_team_name_by_id(team_id):
    nba_teams = teams.get_teams()
    team = next((team for team in nba_teams if team['id'] == team_id), None)
    return team['full_name'] if team else None

# Function to get Opponent Team ID from the Game Logs
def get_opponent_team_id(df):
    # Extract opponent team abbreviations from MATCHUP column
    opponent_abbreviations = df['MATCHUP'].apply(lambda x: x.split(' ')[-1]).unique()
    opponent_team_ids = []
    for abbrev in opponent_abbreviations:
        team_id = get_team_id_by_abbreviation(abbrev)
        if team_id:
            opponent_team_ids.append(team_id)
    if not opponent_team_ids:
        print("No opponent teams found in the game logs.")
        return None
    print("\nRecent Opponent Teams:")
    unique_opponents = list(set(opponent_team_ids))
    for idx, team_id in enumerate(unique_opponents, 1):
        team_name = find_team_name_by_id(team_id)
        print(f"{idx}. {team_name} ({team_id})")
    choice = input("Choose an opponent team number for strength analysis (or press Enter to skip): ").strip()
    if choice.isdigit() and 1 <= int(choice) <= len(unique_opponents):
        return unique_opponents[int(choice) - 1]  # Return Team ID
    return None

# Function to get Opponent Team Name from Team ID
def get_team_name_by_id(team_id):
    team_name = find_team_name_by_id(team_id)
    return team_name if team_name else "Unknown Team"

# Function to fetch recent opponent team stats
def get_opponent_team_stats(team_id, rankings, season="2024-25", num_games=5):
    # Fetch recent team game logs
    logs = teamgamelogs.TeamGameLogs(
        team_id_nullable=team_id,
        season_nullable=season,
        season_type_nullable="Regular Season"  # Correct parameter for TeamGameLogs
    )
    df = logs.get_data_frames()[0]

    if df.empty:
        return None

    # Filter only numeric columns for calculations
    numeric_stats = ['PTS', 'REB', 'AST', 'STL', 'BLK']
    team_games = df[numeric_stats].head(num_games).apply(pd.to_numeric, errors='coerce')

    # Calculate averages for last 'num_games'
    team_avg = team_games.mean()

    # Add rankings
    team_rankings = {stat: rankings[stat].get(team_id, 'N/A') for stat in numeric_stats}

    team_avg = team_avg.round(1).to_dict()
    team_avg.update({f"{stat}_RANK": rank for stat, rank in team_rankings.items()})

    return team_avg

# Function to calculate league-wide team rankings
def get_league_team_rankings(season="2024-25"):
    logging.info("\nFetching league-wide team statistics for rankings...")
    logs = teamgamelogs.TeamGameLogs(
        season_nullable=season,
        season_type_nullable="Regular Season"  # Correct parameter for TeamGameLogs
    )
    df = logs.get_data_frames()[0]

    # Aggregate stats per team
    aggregated = df.groupby('TEAM_ID').agg({
        'PTS': 'mean',
        'REB': 'mean',
        'AST': 'mean',
        'STL': 'mean',
        'BLK': 'mean'
    }).reset_index()

    # Calculate rankings
    stats = ['PTS', 'REB', 'AST', 'STL', 'BLK']
    rankings = {}
    for stat in stats:
        aggregated[f'{stat}_RANK'] = aggregated[stat].rank(ascending=False, method='min')
        rankings[stat] = dict(zip(aggregated['TEAM_ID'], aggregated[f'{stat}_RANK']))

    logging.info("League-wide team rankings calculated.")
    return rankings

# Function to calculate weighted averages of player stats
def calculate_averages(df, num_games=5):
    stats = ['PTS', 'REB', 'AST', 'STL', 'BLK']
    stats = [stat for stat in stats if stat in df.columns]
    df[stats + ['MIN']] = df[stats + ['MIN']].apply(pd.to_numeric, errors='coerce')
    avg_minutes = df['MIN'].mean()

    def weight_game(row):
        if row['MIN'] < 0.7 * avg_minutes:
            return 0.5
        elif row['MIN'] > 0.85 * avg_minutes:
            return 1.5
        return 1.0

    df['WEIGHT'] = df.apply(weight_game, axis=1)
    recent_games = df.head(num_games)
    weighted_avg = (recent_games[stats].multiply(recent_games['WEIGHT'], axis=0)).sum() / recent_games['WEIGHT'].sum()
    p = weighted_avg.get('PTS', 0)
    r = weighted_avg.get('REB', 0)
    a = weighted_avg.get('AST', 0)
    p_r = p + r
    p_a = p + a
    r_a = r + a
    p_r_a = p + r + a
    return {
        'PTS': round(p, 1), 'REB': round(r, 1), 'AST': round(a, 1),
        'STL': round(weighted_avg.get('STL', 0), 1), 'BLK': round(weighted_avg.get('BLK', 0), 1),
        'P+R': round(p_r, 1), 'P+A': round(p_a, 1), 'R+A': round(r_a, 1), 'P+R+A': round(p_r_a, 1)
    }

from nba_api.stats.endpoints import commonteamroster

# Function to get player position with standardized codes
def get_player_position(player_id, season="2024-25"):
    try:
        info = commonplayerinfo.CommonPlayerInfo(player_id=player_id)
        data = info.get_data_frames()[0]
        position_full = data['POSITION'][0].strip()

        # Mapping full position names to shorthand codes
        position_map = {
            'G': 'G',
            'Guard': 'G',
            'F': 'F',
            'Forward': 'F',
            'C': 'C',
            'Center': 'C',
            '': '',
            None: ''
        }

        # Handle cases where 'POSITION' might be in full name or shorthand
        position_shorthand = position_map.get(position_full, '')
        if not position_shorthand:
            # Attempt to derive shorthand from first letter if possible
            if position_full.upper() in ['G', 'F', 'C']:
                position_shorthand = position_full.upper()
            else:
                position_shorthand = ''

        return position_shorthand
    except Exception as e:
        logging.error(f"Error fetching position for player ID {player_id}: {e}")
        return None

# Function to calculate matchup deltas
def get_matchup_deltas(opponent_team_id, position, season="2024-25"):
    logging.info(f"\nCalculating matchup deltas for position: {position} against {get_team_name_by_id(opponent_team_id)}...")
    all_players = players.get_active_players()
    # Safely access 'position' key and compare
    same_position_players = [p for p in all_players if p.get('position', '') == position]

    deltas = {'PTS': [], 'REB': [], 'AST': [], 'STL': [], 'BLK': []}

    opponent_team_name = get_team_name_by_id(opponent_team_id)
    if opponent_team_name is None:
        logging.error("Opponent team name could not be determined.")
        return deltas

    for idx, player in enumerate(same_position_players, 1):
        player_id = player['id']
        try:
            gamelog = playergamelog.PlayerGameLog(
                player_id=player_id,
                season=season,
                season_type_all_star="Regular Season"  # Correct parameter for PlayerGameLog
            )
            df = gamelog.get_data_frames()[0]
            if df.empty:
                continue
            # Games against the opponent
            matchup_games = df[df['MATCHUP'].str.contains(f"@ {opponent_team_name}|vs. {opponent_team_name}")]
            if matchup_games.empty:
                continue
            # Calculate average stats in matchup games
            matchup_avg = matchup_games[['PTS', 'REB', 'AST', 'STL', 'BLK']].mean()
            # Calculate season averages
            season_avg = df[['PTS', 'REB', 'AST', 'STL', 'BLK']].mean()
            # Calculate deltas
            delta = matchup_avg - season_avg
            for stat in deltas.keys():
                deltas[stat].append(delta.get(stat, 0))
        except Exception as e:
            logging.error(f"Error processing player {player['full_name']}: {e}")
            continue

    # Calculate average deltas
    avg_deltas = {stat: round(pd.Series(values).mean(), 2) if values else 0 for stat, values in deltas.items()}
    logging.info("Matchup deltas calculated.")
    return avg_deltas

# Function to adjust projected stats based on matchup deltas
def calculate_projected_stats(projected_line, matchup_deltas):
    projected = {}
    for stat, value in projected_line.items():
        # Only adjust base stats, not combined stats like P+R
        if stat in matchup_deltas:
            projected[stat] = round(value + matchup_deltas[stat], 1)
        else:
            projected[stat] = value
    return projected

# Main Program
if __name__ == "__main__":
    # Fetch league-wide team rankings once to avoid redundant API calls
    try:
        rankings = get_league_team_rankings(season="2024-25")
    except Exception as e:
        logging.error(f"Failed to fetch league team rankings: {e}")
        sys.exit(1)

    while True:
        player_name = input("Enter the player's name or last name (or type 'quit' to exit): ").strip()
        if player_name.lower() in ["quit", "exit"]:
            print("Exiting the program.")
            break

        try:
            num_games_input = input("Enter the number of recent games to analyze (default is 5): ").strip()
            num_games = int(num_games_input) if num_games_input.isdigit() else 5

            season = "2024-25"
            df, full_name, player_id = fetch_player_data_by_name(player_name, season)

            if df.empty:
                print(f"No game logs found for {full_name} in the {season} season.")
                continue

            opponent_team = get_opponent_team_id(df)
            matchup_deltas = {}
            position = None

            if opponent_team:
                opponent_stats = get_opponent_team_stats(opponent_team, rankings, season=season, num_games=num_games)
                if opponent_stats is not None:
                    print("\nOpponent Team Recent Averages and Rankings:")
                    for stat, value in opponent_stats.items():
                        print(f"{stat}: {value}")
                else:
                    print("Could not fetch opponent team stats.")

                # Get player's position
                position = get_player_position(player_id, season=season)
                if position:
                    matchup_deltas = get_matchup_deltas(opponent_team, position, season=season)
                    print("\nMatchup Deltas:")
                    for stat, delta in matchup_deltas.items():
                        print(f"{stat}: {delta}")
                else:
                    print("Could not determine player position.")
            else:
                print("No opponent team selected for strength analysis.")

            # Calculate projected averages
            projected_line = calculate_averages(df, num_games=num_games)

            # Adjust projections based on matchup deltas
            if opponent_team and position and matchup_deltas:
                projected_line = calculate_projected_stats(projected_line, matchup_deltas)

            print(f"\nProjected line for {full_name} based on last {num_games} games and matchup:")
            for stat, value in projected_line.items():
                print(f"{stat}: {value}")

            # Save projections to CSV
            file_name = f"{full_name.replace(' ', '_')}_projected_stats.csv"
            pd.DataFrame([projected_line]).to_csv(file_name, index=False)
            print(f"Projected stats saved to '{file_name}'.")

        except ValueError as e:
            print(e)
        except Exception as e:
            logging.error(f"An error occurred: {e}")
        print("\n--------------------------------------\n")

ERROR:root:Failed to fetch league team rankings: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/urllib3/connection.py", line 516, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/http/client.py", line 1395, in getresponse
    response.begin()
  File "/usr/lib/python3.11/http/client.py", line 325, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/http/client.py", line 286, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/ssl.py", line

TypeError: object of type 'NoneType' has no len()

In [ ]:
!pip install nba_api
from nba_api.stats.endpoints import playergamelog, teamgamelogs, commonplayerinfo
from nba_api.stats.static import players, teams
import pandas as pd
import logging
import sys
import time  # Import time for retry mechanism
import requests # Import requests for retry mechanism

# Setup logging for better visibility
logging.basicConfig(level=logging.INFO, format='%(message)s')

# Function to get Player ID by Partial Name (Active Players Only)
def get_player_id_by_partial_name(player_name):
    all_players = players.get_active_players()
    matching_players = [p for p in all_players if player_name.lower() in p['full_name'].lower()]
    if not matching_players:
        raise ValueError(f"No active player found with the name '{player_name}'.")
    if len(matching_players) > 12:
        print("Too many results found. Showing the first 12 matches:")
        matching_players = matching_players[:12]
    if len(matching_players) > 1:
        print("\nMultiple active players found:")
        for idx, p in enumerate(matching_players, 1):
            print(f"{idx}. {p['full_name']}")
        choice = input("Enter the number of the player you want: ").strip()
        if choice.isdigit() and 1 <= int(choice) <= len(matching_players):
            selected = matching_players[int(choice) - 1]
            return selected['id'], selected['full_name']
        else:
            raise ValueError("Invalid selection. Please try again.")
    else:
        return matching_players[0]['id'], matching_players[0]['full_name']

# Function to fetch Player Game Logs
def fetch_player_data_by_name(player_name, season="2024-25"):
    player_id, full_name = get_player_id_by_partial_name(player_name)
    logging.info(f"\nFetching game logs for: {full_name}")
    gamelog = playergamelog.PlayerGameLog(
        player_id=player_id,
        season=season,
        season_type_all_star="Regular Season"  # Correct parameter for PlayerGameLog
    )
    df = gamelog.get_data_frames()[0]
    return df, full_name, player_id

# Function to map Team Abbreviation to Team ID
def get_team_id_by_abbreviation(abbreviation):
    nba_teams = teams.get_teams()
    team = next((team for team in nba_teams if team['abbreviation'] == abbreviation), None)
    return team['id'] if team else None

# Function to find Team Name by Team ID
def find_team_name_by_id(team_id):
    nba_teams = teams.get_teams()
    team = next((team for team in nba_teams if team['id'] == team_id), None)
    return team['full_name'] if team else None

# Function to get Opponent Team ID from the Game Logs
def get_opponent_team_id(df):
    # Extract opponent team abbreviations from MATCHUP column
    opponent_abbreviations = df['MATCHUP'].apply(lambda x: x.split(' ')[-1]).unique()
    opponent_team_ids = []
    for abbrev in opponent_abbreviations:
        team_id = get_team_id_by_abbreviation(abbrev)
        if team_id:
            opponent_team_ids.append(team_id)
    if not opponent_team_ids:
        print("No opponent teams found in the game logs.")
        return None
    print("\nRecent Opponent Teams:")
    unique_opponents = list(set(opponent_team_ids))
    for idx, team_id in enumerate(unique_opponents, 1):
        team_name = find_team_name_by_id(team_id)
        print(f"{idx}. {team_name} ({team_id})")
    choice = input("Choose an opponent team number for strength analysis (or press Enter to skip): ").strip()
    if choice.isdigit() and 1 <= int(choice) <= len(unique_opponents):
        return unique_opponents[int(choice) - 1]  # Return Team ID
    return None

# Function to get Opponent Team Name from Team ID
def get_team_name_by_id(team_id):
    team_name = find_team_name_by_id(team_id)
    return team_name if team_name else "Unknown Team"

# Function to fetch recent opponent team stats
def get_opponent_team_stats(team_id, rankings, season="2024-25", num_games=5):
    # Fetch recent team game logs
    logs = teamgamelogs.TeamGameLogs(
        team_id_nullable=team_id,
        season_nullable=season,
        season_type_nullable="Regular Season"  # Correct parameter for TeamGameLogs
    )
    df = logs.get_data_frames()[0]

    if df.empty:
        return None

    # Filter only numeric columns for calculations
    numeric_stats = ['PTS', 'REB', 'AST', 'STL', 'BLK']
    team_games = df[numeric_stats].head(num_games).apply(pd.to_numeric, errors='coerce')

    # Calculate averages for last 'num_games'
    team_avg = team_games.mean()

    # Add rankings
    team_rankings = {stat: rankings[stat].get(team_id, 'N/A') for stat in numeric_stats}

    team_avg = team_avg.round(1).to_dict()
    team_avg.update({f"{stat}_RANK": rank for stat, rank in team_rankings.items()})

    return team_avg

# Function to calculate league-wide team rankings
def get_league_team_rankings(season="2024-25"):
    logging.info("\nFetching league-wide team statistics for rankings...")
    logs = teamgamelogs.TeamGameLogs(
        season_nullable=season,
        season_type_nullable="Regular Season"  # Correct parameter for TeamGameLogs
    )
    df = logs.get_data_frames()[0]

    # Aggregate stats per team
    aggregated = df.groupby('TEAM_ID').agg({
        'PTS': 'mean',
        'REB': 'mean',
        'AST': 'mean',
        'STL': 'mean',
        'BLK': 'mean'
    }).reset_index()

    # Calculate rankings
    stats = ['PTS', 'REB', 'AST', 'STL', 'BLK']
    rankings = {}
    for stat in stats:
        aggregated[f'{stat}_RANK'] = aggregated[stat].rank(ascending=False, method='min')
        rankings[stat] = dict(zip(aggregated['TEAM_ID'], aggregated[f'{stat}_RANK']))

    logging.info("League-wide team rankings calculated.")
    return rankings

# Function to calculate weighted averages of player stats
def calculate_averages(df, num_games=5):
    stats = ['PTS', 'REB', 'AST', 'STL', 'BLK']
    stats = [stat for stat in stats if stat in df.columns]
    df[stats + ['MIN']] = df[stats + ['MIN']].apply(pd.to_numeric, errors='coerce')
    avg_minutes = df['MIN'].mean()

    def weight_game(row):
        if row['MIN'] < 0.7 * avg_minutes:
            return 0.5
        elif row['MIN'] > 0.85 * avg_minutes:
            return 1.5
        return 1.0

    df['WEIGHT'] = df.apply(weight_game, axis=0) # corrected axis
    recent_games = df.head(num_games)
    weighted_avg = (recent_games[stats].multiply(recent_games['WEIGHT'], axis=0)).sum() / recent_games['WEIGHT'].sum()
    p = weighted_avg.get('PTS', 0)
    r = weighted_avg.get('REB', 0)
    a = weighted_avg.get('AST', 0)
    p_r = p + r
    p_a = p + a
    r_a = r + a
    p_r_a = p + r + a
    return {
        'PTS': round(p, 1), 'REB': round(r, 1), 'AST': round(a, 1),
        'STL': round(weighted_avg.get('STL', 0), 1), 'BLK': round(weighted_avg.get('BLK', 0), 1),
        'P+R': round(p_r, 1), 'P+A': round(p_a, 1), 'R+A': round(r_a, 1), 'P+R+A': round(p_r_a, 1)
    }


# Function to get player position with standardized codes (Retry Mechanism and Simplified Mapping)
def get_player_position(player_id, season="2024-25", max_retries=3, initial_delay=1): # Added retry parameters
    retries = 0
    delay = initial_delay
    while retries < max_retries:
        try:
            info = commonplayerinfo.CommonPlayerInfo(player_id=player_id)
            data = info.get_data_frames()[0]
            position_full = data['POSITION'][0].strip()

            # Simplified Position Mapping to C, F, G
            position_map = {
                'G': 'G',
                'Guard': 'G',
                'SG': 'G',          # Shooting Guard maps to G
                'Shooting Guard': 'G',
                'PG': 'G',          # Point Guard maps to G
                'Point Guard': 'G',
                'F': 'F',
                'Forward': 'F',
                'SF': 'F',          # Small Forward maps to F
                'Small Forward': 'F',
                'PF': 'F',          # Power Forward maps to F
                'Power Forward': 'F',
                'C': 'C',
                'Center': 'C',
                '': '',
                None: ''
            }

            position_shorthand = position_map.get(position_full, '')
            if not position_shorthand:
                if position_full.upper() in ['G', 'F', 'C']: # Keep simplified check
                    position_shorthand = position_full.upper()
                else:
                    position_shorthand = ''
            return position_shorthand

        except requests.exceptions.ReadTimeout as e: # Catch ReadTimeout specifically
            retries += 1
            logging.warning(f"Timeout fetching position for player ID {player_id}, retry {retries}/{max_retries}. Waiting {delay} seconds...")
            time.sleep(delay)
            delay *= 2  # Exponential backoff (double the delay for each retry)
        except Exception as e:
            logging.error(f"Error fetching position for player ID {player_id} after retries: {e}")
            return None

    logging.error(f"Failed to get position for player ID {player_id} after {max_retries} retries due to timeouts.") # Log failure after all retries
    return None


# Function to calculate matchup deltas (using simplified positions)
def get_matchup_deltas(opponent_team_id, position, season="2024-25"):
    logging.info(f"\nCalculating matchup deltas for position: {position} against {get_team_name_by_id(opponent_team_id)}...")
    all_players = players.get_active_players()
    # Filter players by simplified position (using get_player_position function)
    same_position_players = [
        p for p in all_players
        if get_player_position(p['id'], season=season) == position.upper() # Get simplified position and compare
    ]

    deltas = {'PTS': [], 'REB': [], 'AST': [], 'STL': [], 'BLK': []}

    opponent_team_name = get_team_name_by_id(opponent_team_id)
    if opponent_team_name is None:
        logging.error("Opponent team name could not be determined.")
        return deltas

    logging.info(f"Found {len(same_position_players)} players with position '{position}'.") # logging number of players

    for idx, player in enumerate(same_position_players, 1):
        player_id = player['id']
        player_full_name = player['full_name'] # getting player name for logging
        logging.info(f"Processing player {idx}/{len(same_position_players)}: {player_full_name} (ID: {player_id})") # logging player being processed
        try:
            gamelog = playergamelog.PlayerGameLog(
                player_id=player_id,
                season=season,
                season_type_all_star="Regular Season"  # Correct parameter for PlayerGameLog
            )
            df = gamelog.get_data_frames()[0]
            if df.empty:
                logging.info(f"No game logs found for {player_full_name}. Skipping.") # logging empty game logs
                continue
            # Games against the opponent (case-insensitive)
            matchup_games = df[df['MATCHUP'].str.contains(f"@{opponent_team_name}|vs. {opponent_team_name}", case=False)]
            if matchup_games.empty:
                logging.info(f"No matchup games found for {player_full_name} against {opponent_team_name}. Skipping.") # logging no matchup games
                continue

            logging.info(f"Found {len(matchup_games)} matchup games for {player_full_name} against {opponent_team_name}.") # logging number of matchup games

            # Calculate average stats in matchup games
            matchup_avg = matchup_games[['PTS', 'REB', 'AST', 'STL', 'BLK']].mean()
            # Calculate season averages
            season_avg = df[['PTS', 'REB', 'AST', 'STL', 'BLK']].mean()
            # Calculate deltas
            delta = matchup_avg - season_avg

            logging.info(f"Matchup averages for {player_full_name}: {matchup_avg.to_dict()}") # logging matchup averages
            logging.info(f"Season averages for {player_full_name}: {season_avg.to_dict()}") # logging season averages
            logging.info(f"Deltas for {player_full_name}: {delta.to_dict()}") # logging deltas

            for stat in deltas.keys():
                deltas[stat].append(delta.get(stat, 0))
        except Exception as e:
            logging.error(f"Error processing player {player_full_name}: {e}")
            continue

    # Calculate average deltas
    avg_deltas = {stat: round(pd.Series(values).mean(), 2) if values else 0 for stat, values in deltas.items()}
    logging.info("Matchup deltas calculated.")
    logging.info(f"Average deltas: {avg_deltas}") # logging average deltas
    return avg_deltas

# Function to adjust projected stats based on matchup deltas
def calculate_projected_stats(projected_line, matchup_deltas):
    projected = {}
    for stat, value in projected_line.items():
        # Only adjust base stats, not combined stats like P+R
        if stat in matchup_deltas:
            projected[stat] = round(value + matchup_deltas[stat], 1)
        else:
            projected[stat] = value
    return projected

# Main Program
if __name__ == "__main__":
    # Fetch league-wide team rankings once to avoid redundant API calls
    try:
        rankings = get_league_team_rankings(season="2024-25")
    except Exception as e:
        logging.error(f"Failed to fetch league team rankings: {e}")
        sys.exit(1)

    while True:
        player_name = input("Enter the player's name or last name (or type 'quit' to exit): ").strip()
        if player_name.lower() in ["quit", "exit"]:
            print("Exiting the program.")
            break

        try:
            num_games_input = input("Enter the number of recent games to analyze (default is 5): ").strip()
            num_games = int(num_games_input) if num_games_input.isdigit() else 5

            season = "2024-25"
            df, full_name, player_id = fetch_player_data_by_name(player_name, season)

            if df.empty:
                print(f"No game logs found for {full_name} in the {season} season.")
                continue

            opponent_team = get_opponent_team_id(df)
            matchup_deltas = {}
            position = None

            if opponent_team:
                opponent_stats = get_opponent_team_stats(opponent_team, rankings, season=season, num_games=num_games)
                if opponent_stats is not None:
                    print("\nOpponent Team Recent Averages and Rankings:")
                    for stat, value in opponent_stats.items():
                        print(f"{stat}: {value}")
                else:
                    print("Could not fetch opponent team stats.")

                # Get player's position (simplified)
                position = get_player_position(player_id, season=season)
                if position:
                    print(f"\nPlayer position (simplified): {position}") # Inform user of simplified position

                    # Ask user to confirm or choose position for matchup analysis
                    position_choice = input(f"Analyze matchup for position '{position}'? (yes/no/enter position 'G', 'F', 'C'): ").strip().lower()
                    if position_choice == 'no':
                        position_input = input("Enter position for matchup analysis ('G', 'F', 'C'): ").strip().upper()
                        if position_input in ['G', 'F', 'C']:
                            position_for_deltas = position_input
                        else:
                            print("Invalid position entered, skipping matchup deltas.")
                            position_for_deltas = None # or position, to use the player's position anyway
                    elif position_choice == 'yes' or position_choice == '': # Default to player's position if 'yes' or Enter
                        position_for_deltas = position
                    elif position_choice.upper() in ['G', 'F', 'C']: # User entered a position directly
                        position_for_deltas = position_choice.upper()
                    else:
                        print("Invalid choice, skipping matchup deltas.")
                        position_for_deltas = None

                    if position_for_deltas:
                        matchup_deltas = get_matchup_deltas(opponent_team, position_for_deltas, season=season)
                        print("\nMatchup Deltas (for position: {}):".format(position_for_deltas))
                        for stat, delta in matchup_deltas.items():
                            print(f"{stat}: {delta}")
                    else:
                        print("Matchup delta analysis skipped.")
                else:
                    print("Could not determine player position.")
            else:
                print("No opponent team selected for strength analysis.")

            # Calculate projected averages
            projected_line = calculate_averages(df, num_games=num_games)

            # Adjust projections based on matchup deltas
            if opponent_team and position_for_deltas and matchup_deltas: # use position_for_deltas here
                projected_line = calculate_projected_stats(projected_line, matchup_deltas)

            print(f"\nProjected line for {full_name} based on last {num_games} games and matchup:")
            for stat, value in projected_line.items():
                print(f"{stat}: {value}")

            # Save projections to CSV
            file_name = f"{full_name.replace(' ', '_')}_projected_stats.csv"
            pd.DataFrame([projected_line]).to_csv(file_name, index=False)
            print(f"Projected stats saved to '{file_name}'.")

        except ValueError as e:
            print(e)
        except Exception as e:
            logging.error(f"An error occurred: {e}")
        print("\n--------------------------------------\n")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.9/284.9 kB 5.1 MB/s eta 0:00:00
Enter the player's name or last name (or type 'quit' to exit): lavine
Enter the number of recent games to analyze (default is 5): 5

Recent Opponent Teams:
1. Atlanta Hawks (1610612737)
2. Boston Celtics (1610612738)
3. Cleveland Cavaliers (1610612739)
4. New Orleans Pelicans (1610612740)
5. Chicago Bulls (1610612741)
6. Dallas Mavericks (1610612742)
7. Denver Nuggets (1610612743)
8. Golden State Warriors (1610612744)
9. Houston Rockets (1610612745)
10. Los Angeles Clippers (1610612746)
11. Milwaukee Bucks (1610612749)
12. Brooklyn Nets (1610612751)
13. New York Knicks (1610612752)
14. Orlando Magic (1610612753)
15. Indiana Pacers (1610612754)
16. Philadelphia 76ers (1610612755)
17. Phoenix Suns (1610612756)
18. Portland Trail Blazers (1610612757)
19. Sacramento Kings (1610612758)
20. San Antonio Spurs (1610612759)
21. Oklahoma City Thunder (1610612760)
22. Utah Jazz (1610612762)
23. Memphis Grizzlies (1610

KeyboardInterrupt: 